In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUTS = Path('../outputs/swift_apex_benchmark')

rows = []
for result_file in sorted(OUTPUTS.rglob('result_*.json')):
    parts = result_file.parts
    method = parts[-4]
    task   = parts[-3]
    seed   = parts[-2].replace('seed_', '')
    with open(result_file, encoding='utf-8') as f:
        d = json.load(f)
    rows.append({
        'method':       method,
        'task':         task.replace('bbh_', ''),
        'seed':         int(seed),
        'dev_score':    d.get('best_score', 0.0),
        'test_score':   d.get('test_score', 0.0),
        'total_time':   d.get('total_time', 0.0),
        'llm_calls':    d.get('llm_usage', {}).get('total_calls', 0),
        'total_tokens': d.get('llm_usage', {}).get('total_tokens', 0),
        'num_iters':    d.get('num_iterations', 0),
    })

df = pd.DataFrame(rows)
print(f'{len(df)} runs loaded — methods: {sorted(df.method.unique())} | tasks: {sorted(df.task.unique())}')
df.head()

104 runs loaded — methods: ['apex', 'capo', 'gaapo', 'see', 'swift'] | tasks: ['causal_judgement', 'disambiguation_qa', 'formal_fallacies', 'hyperbaton', 'logical_deduction_five_objects', 'penguins_in_a_table', 'reasoning_about_colored_objects']


,method,task,seed,dev_score,test_score,total_time,llm_calls,total_tokens,num_iters
0,apex,causal_judgement,123,0.625,0.60,119.45,559,284335,4
1,apex,causal_judgement,42,0.650,0.50,145.38,723,348007,5
2,apex,causal_judgement,7,0.775,0.45,141.11,753,359365,5
3,apex,disambiguation_qa,123,0.475,0.30,225.86,843,224067,5
4,apex,disambiguation_qa,42,0.475,0.50,238.83,870,260470,5


## Test Score — mean ± std across seeds

In [2]:
agg = df.groupby(['method', 'task'])['test_score'].agg(['mean', 'std', 'count']).reset_index()
agg['std'] = agg['std'].fillna(0.0)
agg['score'] = agg.apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1)

pivot = agg.pivot(index='task', columns='method', values='score')

# Also compute a mean column per method (macro-average over tasks)
mean_per_method = df.groupby('method')['test_score'].mean()
macro_row = {m: f"{mean_per_method[m]:.3f}" for m in pivot.columns}
pivot.loc['**macro avg**'] = macro_row

pd.set_option('display.max_colwidth', 20)
print('Test score (mean ± std, 3 seeds)')
pivot

Test score (mean ± std, 3 seeds)


method,apex,capo,gaapo,see,swift
task,,,,,
causal_judgement,0.517 ± 0.076,0.467 ± 0.118,0.567 ± 0.038,0.475 ± 0.139,0.550 ± 0.066
disambiguation_qa,0.383 ± 0.104,0.358 ± 0.151,0.025 ± 0.043,0.358 ± 0.128,0.417 ± 0.052
formal_fallacies,0.692 ± 0.095,0.650 ± 0.066,0.692 ± 0.058,0.692 ± 0.038,0.608 ± 0.063
hyperbaton,0.450 ± 0.303,0.800 ± 0.025,0.558 ± 0.397,0.717 ± 0.104,0.667 ± 0.052
logical_deduction_five_objects,0.233 ± 0.274,0.608 ± 0.088,0.000 ± 0.000,0.550 ± 0.050,0.525 ± 0.090
penguins_in_a_table,0.367 ± 0.275,0.500 ± 0.100,0.125 ± 0.139,0.517 ± 0.038,0.383 ± 0.080
reasoning_about_colored_objects,0.250 ± 0.229,0.562 ± 0.088,0.000 ± 0.000,0.500 ± 0.087,0.458 ± 0.288
**macro avg**,0.413,0.564,0.281,0.544,0.515


## Dev vs Test Score (overfitting check)

In [3]:
gap = df.groupby(['method', 'task']).agg(
    dev_mean=('dev_score', 'mean'),
    test_mean=('test_score', 'mean'),
).reset_index()
gap['gap'] = gap['dev_mean'] - gap['test_mean']

gap_pivot = gap.pivot(index='task', columns='method', values='gap').round(3)
print('Dev - Test gap (positive = overfit to dev)')
gap_pivot.style.background_gradient(cmap='RdYlGn_r', axis=None)

Dev - Test gap (positive = overfit to dev)


ImportError: `Import matplotlib` failed. Styler.background_gradient requires matplotlib. Use pip or conda to install the matplotlib package.

## Efficiency — LLM calls and wall-clock time per run

In [ ]:
eff = df.groupby('method').agg(
    avg_calls=('llm_calls', 'mean'),
    avg_tokens=('total_tokens', 'mean'),
    avg_time_s=('total_time', 'mean'),
    avg_iters=('num_iters', 'mean'),
    avg_test_score=('test_score', 'mean'),
).round(1)
eff['score_per_100_calls'] = (eff['avg_test_score'] / (eff['avg_calls'] / 100)).round(3)
eff

## Per-task best method

In [ ]:
best = agg[agg['task'] != '**macro avg**'].copy()
best['mean_val'] = best['mean']
idx = best.groupby('task')['mean_val'].idxmax()
best_per_task = best.loc[idx, ['task', 'method', 'mean', 'std']].rename(
    columns={'mean': 'test_mean', 'std': 'test_std', 'method': 'best_method'}
).set_index('task')
best_per_task

## Raw data

In [ ]:
df.sort_values(['task', 'method', 'seed']).reset_index(drop=True)